In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import datetime



# Input ROOT file
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20250310.root"

# Open the ROOT file with uproot
with uproot.open(input_filename) as file:
    tree = file["resampled_data"]
    data = tree.arrays(library="np")

# Extract times and wav values
times = data["t"]
wav0 = data["wav"][:, 0, 0]  # Polarization s
wav1 = data["wav"][:, 1, 0]  # Polarization p

# Convert times to datetime
timestamps = np.array([datetime.datetime.utcfromtimestamp(float(t)) for t in times])

# Filter by wavelength threshold
wavelength_min = 1.5574e-6
wavelength_max = 1.5580e-6
mask0 = (wav0 > wavelength_min) & (wav0 < wavelength_max)
mask1 = (wav1 > wavelength_min) & (wav1 < wavelength_max)

# Apply masks
timestamps0 = timestamps[mask0]
wav0_filtered = wav0[mask0]

timestamps1 = timestamps[mask1]
wav1_filtered = wav1[mask1]

# --- Filter by time range ---
start_time = datetime.datetime(2025, 3, 10, 13, 45, 0)
end_time = datetime.datetime(2025, 3, 10, 14, 15, 0)

time_mask0 = (timestamps0 >= start_time) & (timestamps0 <= end_time)
time_mask1 = (timestamps1 >= start_time) & (timestamps1 <= end_time)

timestamps0 = timestamps0[time_mask0]
wav0_filtered = wav0_filtered[time_mask0]

timestamps1 = timestamps1[time_mask1]
wav1_filtered = wav1_filtered[time_mask1]

# --- Convert to picometers and subtract first value ---
wav0_pm = (wav0_filtered - wav0_filtered[0]) * 1e12
wav1_pm = (wav1_filtered - wav0_filtered[0]) * 1e12

# --- Calculate mean polarization in pm ---
common_times = np.intersect1d(timestamps0, timestamps1)
wav0_common = np.interp([t.timestamp() for t in common_times], 
                        [t.timestamp() for t in timestamps0], wav0_pm)
wav1_common = np.interp([t.timestamp() for t in common_times], 
                        [t.timestamp() for t in timestamps1], wav1_pm)
wav_mean = (wav0_common + wav1_common) / 2

# --- Plot ---
plt.figure(figsize=(12, 6))
# s and p only markers
plt.scatter(timestamps0, wav0_pm, c='blue', marker='o', label='S Pol.')
plt.scatter(timestamps1, wav1_pm, c='green', marker='o', label='P Pol.')
# mean polarization with markers and lines
plt.plot(common_times, wav_mean, '-o', color='red', label='Avg. Pol.')

plt.xlabel("Time (UTC)", fontsize=14)
plt.ylabel("Wavelength (pm)", fontsize=14)
plt.title("Polarization Comparison", fontsize=16)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import uproot
import numpy as np
import matplotlib.pyplot as plt
import datetime

# --- Archivo ROOT ---
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/resampled20250310.root"

# --- Abrir archivo y extraer datos ---
with uproot.open(input_filename) as file:
    tree = file["resampled_data"]
    data = tree.arrays(library="np")

# --- Extraer tiempos ---
times = data["t"]
timestamps = np.array([datetime.datetime.utcfromtimestamp(float(t)) for t in times])

# --- Datos ---
sensor_temp_id = 7  # último sensor
temp_all = data["temp"][:, sensor_temp_id]

# Wavelength (en metros inicialmente)
wav_all = data["wav"][:, 0, 0]
umbral_min = 1.5572e-6
umbral_max = 1.5580e-6

# --- Shift de 1 hora en los picos ---
timestamps_wav_shifted = np.array([t + datetime.timedelta(hours=1) for t in timestamps])

# --- Filtrar por intervalo de tiempo ---
start_time = datetime.datetime.strptime("2025-03-10 13:30:00", "%Y-%m-%d %H:%M:%S")
end_time   = datetime.datetime.strptime("2025-03-10 17:09:00", "%Y-%m-%d %H:%M:%S")

mask_temp = (timestamps >= start_time) & (timestamps <= end_time)
mask_wav  = (timestamps_wav_shifted >= start_time) & (timestamps_wav_shifted <= end_time) & \
            (wav_all >= umbral_min) & (wav_all <= umbral_max)

# --- Aplicar máscaras ---
timestamps_temp_common = timestamps[mask_temp]
temp_common = temp_all[mask_temp]

timestamps_wav_common = timestamps_wav_shifted[mask_wav]
wav_common = wav_all[mask_wav] * 1e12  # Convertir a picómetros (pm)

# --- Plot combinado ---
fig, ax1 = plt.subplots(figsize=(12,6))

color_temp = 'tab:red'
color_wav  = 'tab:blue'

# Temperatura
ax1.set_xlabel("Time (UTC)", fontsize=14, fontweight='bold')
ax1.set_ylabel("Temperature (K)", color=color_temp, fontsize=14, fontweight='bold')
ax1.plot(timestamps_temp_common, temp_common, marker='o', linestyle='-', color=color_temp)
ax1.tick_params(axis='y', labelcolor=color_temp, labelsize=12)
ax1.tick_params(axis='x', labelsize=12)

# Wavelength
ax2 = ax1.twinx()
ax2.set_ylabel("Wavelength (pm)", color=color_wav, fontsize=14, fontweight='bold')
ax2.plot(timestamps_wav_common, wav_common, marker='x', linestyle='-', color=color_wav)
ax2.tick_params(axis='y', labelcolor=color_wav, labelsize=12)

# --- Estética ---
fig.autofmt_xdate(rotation=45)
fig.tight_layout()
ax1.grid(True, alpha=0.3)
plt.title("Temperature and Wavelength vs Time", fontsize=16, fontweight='bold')
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
import datetime
import uproot

# -------------------------
# PARAMETERS
# -------------------------
input_filename = "/eos/user/j/jcapotor/FBGdata/ROOTFiles/pressure_setup/20250515.root"
start_time = datetime.datetime.strptime("2025-05-15 12:30:00", "%Y-%m-%d %H:%M:%S")

sensor_temp = 7
ormocer_sensor_idx = 2
peek_sensor_idx = 4
pol_index = 0

params = {
    "press": {"tolerance": 0.25, "min_plateau_length": 1000},
    "temp":  {"tolerance": 0.5,  "min_plateau_length": 800},
    "wav":   {"tolerance": 5e-6, "min_plateau_length": 800}
}

# -------------------------
# HELPER FUNCTIONS
# -------------------------
def find_plateaus(values, times, tolerance=0.01, min_plateau_length=500):
    plateaus = []
    start_idx = 0
    for i in range(1, len(values)):
        if abs(values[i] - values[start_idx]) > tolerance:
            duration = times[i - 1] - times[start_idx]
            if duration >= min_plateau_length:
                segment = values[start_idx:i]
                plateaus.append({
                    "t0": times[start_idx],
                    "tfin": times[i - 1],
                    "mean": np.mean(segment),
                    "std": np.std(segment),
                    "data": segment
                })
            start_idx = i
    return plateaus

def calculate_plateaus(input_filename, sensor_temp=0, sensor_wav=(0,0),
                       params=None, start_time=None):
    if params is None:
        params = {
            "press": {"tolerance": 0.01, "min_plateau_length": 500},
            "temp":  {"tolerance": 0.01, "min_plateau_length": 500},
            "wav":   {"tolerance": 0.01, "min_plateau_length": 500}
        }
    with uproot.open(input_filename) as f:
        peak_data = f["peak"].arrays(library="np")
        press_data = f["press"].arrays(library="np")
        temp_data = f["temp"].arrays(library="np")

    peak_times = np.array([datetime.datetime.utcfromtimestamp(t) + datetime.timedelta(hours=2)
                           for t in peak_data["t"][:,0]])
    press_times = np.array([datetime.datetime.utcfromtimestamp(t) for t in press_data["t"]])
    temp_times = np.array([datetime.datetime.utcfromtimestamp(t) for t in temp_data["t"]])

    mask_peak  = peak_times >= start_time if start_time is not None else np.ones_like(peak_times, dtype=bool)
    mask_press = press_times >= start_time if start_time is not None else np.ones_like(press_times, dtype=bool)
    mask_temp  = temp_times >= start_time if start_time is not None else np.ones_like(temp_times, dtype=bool)

    peak_values = peak_data["wav"][mask_peak][:, pol_index, sensor_wav[1]] if isinstance(sensor_wav, tuple) else None
    press_values = press_data["press"][mask_press]
    temp_values = temp_data["temp"][mask_temp, sensor_temp]

    press_plateaus = find_plateaus(press_values, np.array([t.timestamp() for t in press_times[mask_press]]), **params["press"])
    temp_plateaus = find_plateaus(temp_values, np.array([t.timestamp() for t in temp_times[mask_temp]]), **params["temp"])
    wav_plateaus = []

    return {
        "peak_data": peak_data,
        "temp_data": temp_data,
        "press_data": press_data,
        "peak_times": peak_times,
        "temp_times": temp_times,
        "press_times": press_times,
        "mask_peak": mask_peak,
        "mask_temp": mask_temp,
        "mask_press": mask_press,
        "press_plateaus": press_plateaus,
        "temp_plateaus": temp_plateaus,
        "wav_plateaus": wav_plateaus
    }

# -------------------------
# LOAD DATA
# -------------------------
data_dict = calculate_plateaus(input_filename,
                               sensor_temp=sensor_temp,
                               sensor_wav=(pol_index, ormocer_sensor_idx),
                               params=params,
                               start_time=start_time)

peak_data = data_dict["peak_data"]
temp_data = data_dict["temp_data"]
peak_times = data_dict["peak_times"]
temp_times = data_dict["temp_times"]

# -------------------------
# COMMON PLATEAUS (manual)
# -------------------------
common_plateaus = [
    {"t0": datetime.datetime(2025,5,15,12,45,52).timestamp(), "tfin": datetime.datetime(2025,5,15,12,59,50).timestamp()},
    {"t0": datetime.datetime(2025,5,15,13,1,19).timestamp(),  "tfin": datetime.datetime(2025,5,15,13,17,42).timestamp()},
    {"t0": datetime.datetime(2025,5,15,13,20,12).timestamp(), "tfin": datetime.datetime(2025,5,15,13,35,16).timestamp()},
    {"t0": datetime.datetime(2025,5,15,13,41,7).timestamp(),  "tfin": datetime.datetime(2025,5,15,13,56,20).timestamp()},
    {"t0": datetime.datetime(2025,5,15,14,1,41).timestamp(),  "tfin": datetime.datetime(2025,5,15,14,12,16).timestamp()},
    {"t0": datetime.datetime(2025,5,15,14,20,54).timestamp(), "tfin": datetime.datetime(2025,5,15,14,36,24).timestamp()},
    {"t0": datetime.datetime(2025,5,15,14,44,54).timestamp(), "tfin": datetime.datetime(2025,5,15,14,46,24).timestamp()},
    {"t0": datetime.datetime(2025,5,15,14,50,54).timestamp(), "tfin": datetime.datetime(2025,5,15,14,52,24).timestamp()}
]

# -------------------------
# EXTRACT PLATEAU MEANS + UNIT CHECKS
# -------------------------
def extract_plateau_means(sensor_idx):
    t_means, t_stds = [], []
    wav_means, wav_stds = [], []
    for p in common_plateaus:
        t0 = datetime.datetime.fromtimestamp(p["t0"])
        tfin = datetime.datetime.fromtimestamp(p["tfin"])
        temp_idx = np.where((temp_times >= t0) & (temp_times <= tfin))[0]
        wav_idx  = np.where((peak_times >= t0) & (peak_times <= tfin))[0]
        if len(temp_idx) == 0 or len(wav_idx) == 0:
            continue
        temp_vals = temp_data["temp"][temp_idx, sensor_temp]
        wav_vals = peak_data["wav"][wav_idx, pol_index, sensor_idx]  # µm

        # 🔎 Print de diagnóstico
        print(f"\n[{sensor_idx}] Plateau {p['t0']}–{p['tfin']}")
        print(f"  wav_vals (µm): min={wav_vals.min():.9f}, max={wav_vals.max():.9f}")
        print(f"  mean={np.mean(wav_vals):.9f}, std={np.std(wav_vals):.2e}")

        t_means.append(np.mean(temp_vals))
        t_stds.append(np.std(temp_vals))
        wav_means.append(np.mean(wav_vals))
        wav_stds.append(np.std(wav_vals))
    return np.array(t_means), np.array(t_stds), np.array(wav_means), np.array(wav_stds)

t_mean_orm, t_std_orm, wav_mean_orm, wav_std_orm = extract_plateau_means(ormocer_sensor_idx)
t_mean_peek, t_std_peek, wav_mean_peek, wav_std_peek = extract_plateau_means(peek_sensor_idx)

# 🔎 Unidades iniciales
print("\n--- UNIT CHECK: raw wav_means ---")
print(f"ORMOCER: {wav_mean_orm.min():.9f} – {wav_mean_orm.max():.9f} µm")
print(f"PEEK:    {wav_mean_peek.min():.9f} – {wav_mean_peek.max():.9f} µm")

# -------------------------
# AUTO UNIT DETECTION
# -------------------------
def detect_unit_and_convert(arr):
    if arr.mean() > 1e-3:      # ej. 1.55 → micrómetros
        print("→ interpretado como µm → convertido a metros")
        return arr * 1e-6
    elif arr.mean() > 1e-6:    # ej. 1.55e-6 → metros
        print("→ interpretado como metros → no se convierte")
        return arr
    elif arr.mean() > 1e-9:    # ej. 1550e-9 → nanómetros
        print("→ interpretado como nm → convertido a metros")
        return arr * 1e-9
    else:
        print("⚠️ valor muy pequeño, revisa las unidades")
        return arr

wav_mean_orm_m = detect_unit_and_convert(wav_mean_orm)
wav_mean_peek_m = detect_unit_and_convert(wav_mean_peek)
wav_std_orm_m = detect_unit_and_convert(wav_std_orm)
wav_std_peek_m = detect_unit_and_convert(wav_std_peek)


# 🔎 Post-conversion check
print("\n--- UNIT CHECK: converted to meters ---")
print(f"ORMOCER: {wav_mean_orm_m.min():.3e} – {wav_mean_orm_m.max():.3e} m")
print(f"PEEK:    {wav_mean_peek_m.min():.3e} – {wav_mean_peek_m.max():.3e} m")

# -------------------------
# CENTER AT LAST POINT (~Tmax)
# -------------------------
wav_orm_centered = wav_mean_orm_m - wav_mean_orm_m[-1]
wav_peek_centered = wav_mean_peek_m - wav_mean_peek_m[-1]

# 🔎 Centering check
print("\n--- CENTERING CHECK ---")
print(f"ORMOCER last point (m): {wav_mean_orm_m[-1]:.6e}")
print(f"PEEK last point (m):    {wav_mean_peek_m[-1]:.6e}")
print(f"ORMOCER Δλ range (m): {wav_orm_centered.min():.2e} – {wav_orm_centered.max():.2e}")
print(f"PEEK Δλ range (m):    {wav_peek_centered.min():.2e} – {wav_peek_centered.max():.2e}")

# -------------------------
# FIT RELATIVE TO LAST PLATEAU (Y=0)
# -------------------------
slope_orm, _, _, _, _ = linregress(t_mean_orm, wav_orm_centered)
slope_peek, _, _, _, _ = linregress(t_mean_peek, wav_peek_centered)

print("\n--- FIT CHECK ---")
print(f"ORMOCER slope: {slope_orm:.3e} m/K = {slope_orm*1e12:.2f} pm/K")
print(f"PEEK slope:    {slope_peek:.3e} m/K = {slope_peek*1e12:.2f} pm/K")

# -------------------------
# CONVERT TO pm (for plotting)
# -------------------------
wav_orm_centered_pm = wav_orm_centered * 1e12
wav_peek_centered_pm = wav_peek_centered * 1e12
wav_std_orm_pm = wav_std_orm_m * 1e12
wav_std_peek_pm = wav_std_peek_m * 1e12
fit_orm_pm = slope_orm * (t_mean_orm - t_mean_orm[-1]) * 1e12
fit_peek_pm = slope_peek * (t_mean_peek - t_mean_peek[-1]) * 1e12

# 🔎 Final units check
print("\n--- FINAL CHECK (for plot) ---")
print(f"ORMOCER Δλ range (pm): {wav_orm_centered_pm.min():.2f} – {wav_orm_centered_pm.max():.2f}")
print(f"PEEK Δλ range (pm):    {wav_peek_centered_pm.min():.2f} – {wav_peek_centered_pm.max():.2f}")

# -------------------------
# PLOT
# -------------------------
plt.figure(figsize=(10,7))

plt.errorbar(t_mean_orm, wav_orm_centered_pm, xerr=t_std_orm, yerr=wav_std_orm_pm,
             fmt='o', color='royalblue', ecolor='gray', capsize=3, markersize=5, label='_nolegend_')
plt.errorbar(t_mean_peek, wav_peek_centered_pm, xerr=t_std_peek, yerr=wav_std_peek_pm,
             fmt='o', color='darkorange', ecolor='gray', capsize=3, markersize=5, label='_nolegend_')

plt.plot(t_mean_orm, fit_orm_pm, 'b--', label=f'ORMOCER fit: {slope_orm*1e12:.1f} pm/mK')
plt.plot(t_mean_peek, fit_peek_pm, 'r--', label=f'PEEK-200µm fit: {slope_peek*1e12:.1f} pm/K')

plt.xlabel("Temperature [K]", fontsize=14, fontweight='bold')
plt.ylabel("Wavelength [pm]", fontsize=14, fontweight='bold')
plt.title("Pressure Setup: PEEK-200µm & ORMOCER", fontsize=15, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
print(t_mean_orm)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

# --- DATASETS ---
t_mean_orm, t_std_orm = np.array(t_mean_orm), np.array(t_std_orm)
t_mean_peek, t_std_peek = np.array(t_mean_peek), np.array(t_std_peek)
wav_mean_orm, wav_std_orm = np.array(wav_mean_orm), np.array(wav_std_orm)
wav_mean_peek, wav_std_peek = np.array(wav_mean_peek), np.array(wav_std_peek)

# --- PEEK-450 µm ---
temp_means_450 = np.array([91.90614259693876, 90.92582215600001, 89.33721636394557,
                           87.4997579954955, 85.63641709090909, 83.406847234375, 
                           80.58533359574467, 76.559555])
temp_stds_450 = np.array([0.0320529342251553, 0.05023405330038183, 0.11030918328238626,
                          0.06149406091283756, 0.03607945982770781, 0.041897072167149864,
                          0.07876260279659503, 0.081897072167149864])
wav_means_450 = np.array([1.5578983712375516, 1.5578772077131646, 1.5578457143627795,
                          1.5578120283047658, 1.5577764377983838, 1.5577306974468623,
                          1.5576775807592511, 1.5576069621333737])
wav_stds_450 = np.array([2.4607612515048697e-06, 2.115290986510928e-06, 2.5140207526671514e-06,
                         2.1925595017914933e-06, 1.8915196228840939e-06, 1.214082553881994e-06,
                         3.357630271068886e-06, 4.157353735686624e-06])

# --- CONVERT WAVES TO METERS ---
datasets = {
    "ORMOCER": (wav_mean_orm, wav_std_orm),
    "PEEK-200µm": (wav_mean_peek, wav_std_peek),
    "PEEK-450µm": (wav_means_450 * 1e-6, wav_stds_450 * 1e-6)  # µm -> m
}

# Conversión
wav_converted = {name: vals * 1.0 for name, (vals, _) in datasets.items()}
wav_std_converted = {name: stds * 1.0 for name, (_, stds) in datasets.items()}

# --- SHIFT WAVES TO LAST POINT (Δλ=0) ---
wav_shifted = {name: vals - vals[-1] for name, vals in wav_converted.items()}

# --- LINEAR FITS ---
fits_pm, slopes_pm = {}, {}
for name in ["ORMOCER", "PEEK-200µm", "PEEK-450µm"]:
    temps = t_mean_orm if name=="ORMOCER" else t_mean_peek if name=="PEEK-200µm" else temp_means_450
    temps_centered = temps - temps[-1]
    slope, _, _, _, _ = linregress(temps_centered, wav_shifted[name])
    fits_pm[name] = slope * temps_centered * 1e12  # pm
    slopes_pm[name] = slope * 1e12  # pm/K
    print(f"{name} slope = {slope:.3e} m/K -> {slopes_pm[name]:.2f} pm/K")

# --- CONVERT ERRORS TO pm ---
wav_std_orm_pm = wav_std_orm * 1e12
wav_std_peek_pm = wav_std_peek * 1e12
wav_std_450_pm = wav_stds_450 * 1e6  # porque wav_450 estaba en m*1e-6 → pm ya lo ajusto a µm? Mejor poner pm

print("--- ERROR CHECK (pm) ---")
print("ORMOCER std (pm):", wav_std_orm_pm)
print("PEEK-200µm std (pm):", wav_std_peek_pm)
print("PEEK-450µm std (pm):", wav_std_450_pm)

# --- PLOT ---
plt.figure(figsize=(10,7))

plt.errorbar(t_mean_orm, wav_shifted["ORMOCER"]*1e12,
             xerr=t_std_orm, yerr=wav_std_orm_pm,
             fmt='o', color='tab:blue', ecolor='gray', capsize=3, markersize=5,
             label=f'ORMOCER fit: {slopes_pm["ORMOCER"]:.2f} pm/K')
plt.plot(t_mean_orm, fits_pm["ORMOCER"], '--', color='tab:blue')

plt.errorbar(t_mean_peek, wav_shifted["PEEK-200µm"]*1e12,
             xerr=t_std_peek, yerr=wav_std_peek_pm,
             fmt='o', color='tab:orange', ecolor='gray', capsize=3, markersize=5,
             label=f'PEEK-200µm fit: {slopes_pm["PEEK-200µm"]:.2f} pm/K')
plt.plot(t_mean_peek, fits_pm["PEEK-200µm"], '--', color='tab:orange')

plt.errorbar(temp_means_450, wav_shifted["PEEK-450µm"]*1e12,
             xerr=temp_stds_450, yerr=wav_std_450_pm,
             fmt='o', color='tab:green', ecolor='gray', capsize=3, markersize=5,
             label=f'PEEK-450µm fit: {slopes_pm["PEEK-450µm"]:.2f} pm/K')
plt.plot(temp_means_450, fits_pm["PEEK-450µm"], '--', color='tab:green')

plt.xlabel("Temperature [K]", fontsize=14, fontweight='bold')
plt.ylabel("Wavelength Diff. [pm]", fontsize=14, fontweight='bold')
plt.title("Pressure Setup: PEEK-200µm & ORMOCER & PEEK-450µm", fontsize=15, fontweight='bold')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
